<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/06_Ajuste_ponderado_covarianza_y_chi2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 06 — Ajuste ponderado, matriz de covarianza y χ² reducido

**Laboratorio 1 · Clase 6**

**Objetivos.**

1. Ajustar usando **las incertezas reales de cada punto**, y entender por qué el ajuste sin pesos es
   el ajuste equivocado cuando los errores son desiguales (O6.2).
2. Obtener las incertezas de los parámetros de la **matriz de covarianza**, y leer la correlación
   entre ellos (O6.3).
3. Usar `absolute_sigma=True` y saber qué hace scipy con el default (O6.4).
4. Usar el **$\chi^2$ reducido** y su p-valor como diagnóstico, incluidos los dos casos patológicos
   que casi nunca se enseñan (O6.5).
5. Reconocer el **promedio ponderado como el ajuste a un modelo constante**, y usar su $\chi^2_\nu$
   como test de consistencia (O6.6).

**Requisitos previos:** Colabs 01 a 05.

Éste es el notebook que vas a volver a abrir todo el resto del cuatrimestre.

> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

np.random.seed(20260916)

def recta(x, a, b):
    return a*x + b

---
## 1. Por qué ponderar

En un laboratorio real, los puntos **no** tienen todos la misma incerteza. Ejemplos típicos:

- un instrumento con error relativo constante: los valores grandes tienen error absoluto mayor;
- un punto medido en el límite del rango del sensor;
- un punto promediado sobre 20 repeticiones y otro sobre 3.

Si algunos puntos son diez veces más confiables que otros y los tratás igual, estás tirando
información. Cuadrados mínimos **ponderados** minimiza

$$ \\chi^2 = \\sum_{i=1}^{N} \\frac{\\left[y_i - f(x_i)\\right]^2}{\\sigma_i^2} $$

es decir, cada término pesa $1/\\sigma_i^2$: cuanto más confiable el punto, más manda.

In [ ]:
# --- DATOS DE EJEMPLO: el resorte del Colab 05, ahora con CUATRO REPETICIONES por punto -----
#     La barra de error se MIDE (dispersión de las repeticiones), no se estima. Y sale distinta
#     para cada punto, porque a masa alta el indicador oscila más y cuesta más leerlo.
masa  = np.array([0.050, 0.100, 0.150, 0.200, 0.250, 0.300, 0.350, 0.400])   # kg
elong = np.array([ 2.74,  6.89, 11.02, 15.10, 19.46, 23.48, 27.10, 29.42])   # cm
err   = np.array([ 0.10,  0.12,  0.15,  0.20,  0.30,  0.45,  0.65,  0.90])   # cm — SEM de 4 repeticiones
# -------------------------------------------------------------------------------------------

p_sin, c_sin = curve_fit(recta, masa, elong)                                 # sin pesos
p_con, c_con = curve_fit(recta, masa, elong, sigma=err, absolute_sigma=True) # ponderado
e_sin, e_con = np.sqrt(np.diag(c_sin)), np.sqrt(np.diag(c_con))

print(f"{'':<12}{'pendiente [cm/kg]':>22}{'ordenada [cm]':>22}")
print(f"{'sin pesos':<12}{p_sin[0]:>12.3f} ± {e_sin[0]:<7.3f}{p_sin[1]:>12.3f} ± {e_sin[1]:<7.3f}")
print(f"{'ponderado':<12}{p_con[0]:>12.3f} ± {e_con[0]:<7.3f}{p_con[1]:>12.3f} ± {e_con[1]:<7.3f}")
print()
g = 9.81
k_sin = g/(p_sin[0]/100); k_con = g/(p_con[0]/100)
print(f"k sin pesos = {k_sin:.2f} N/m     k ponderado = {k_con:.2f} N/m")
print(f"la diferencia es {abs(k_sin-k_con)/(k_con*e_con[0]/p_con[0]):.1f} veces la incerteza de k")

### El argumento `absolute_sigma`

Éste es el detalle que se olvida y arruina el resultado. `curve_fit` tiene dos modos:

- **`absolute_sigma=True`** — toma tus $\\sigma_i$ como **incertezas físicas reales**. La matriz de
  covarianza que devuelve refleja esas incertezas. **Es lo que corresponde en un laboratorio**,
  donde las barras de error salen de la apreciación del instrumento o de la estadística de
  repeticiones.
- **`absolute_sigma=False`** (el valor por defecto) — toma tus $\\sigma_i$ solo como **pesos
  relativos** y reescala la covarianza para que el $\\chi^2$ reducido dé exactamente 1.

El segundo modo es cómodo si no tenés idea de la escala de tus errores, pero tiene una consecuencia
grave: **destruye la información de bondad de ajuste**. Si el código fuerza $\\chi^2_\\nu = 1$, ya no
podés usar $\\chi^2_\\nu$ para saber si el modelo describe los datos.

> **Regla del curso: si tenés barras de error de verdad, `absolute_sigma=True`. Siempre.**

In [ ]:
_, c_rel = curve_fit(recta, masa, elong, sigma=err, absolute_sigma=False)
e_rel = np.sqrt(np.diag(c_rel))

chi2 = np.sum(((elong - recta(masa, *p_con))/err)**2)
nu   = len(masa) - 2
chi2r = chi2 / nu

print(f"errores con absolute_sigma=True  : {e_con}")
print(f"errores con absolute_sigma=False : {e_rel}")
print(f"cociente False/True              : {e_rel/e_con}")
print(f"√(χ²_ν) del ajuste               : {np.sqrt(chi2r):.6f}   <- es EXACTAMENTE el cociente")
print()

# Ahora el caso que importa: alguien que subestima sus barras por un factor 3
err_mal = err / 3
p_m, c_mT = curve_fit(recta, masa, elong, sigma=err_mal, absolute_sigma=True)
_,   c_mF = curve_fit(recta, masa, elong, sigma=err_mal, absolute_sigma=False)
chi2r_mal = np.sum(((elong - recta(masa, *p_m))/err_mal)**2) / nu

print(f"Con las barras subestimadas 3 veces:  χ²_ν = {chi2r_mal:.1f}")
print(f"   σ_pendiente con absolute_sigma=True  : {np.sqrt(c_mT[0,0]):.3f}  <- refleja lo que declaraste")
print(f"   σ_pendiente con absolute_sigma=False : {np.sqrt(c_mF[0,0]):.3f}  <- reescalado por √χ²_ν")
print(f"   el default te DEVUELVE el error correcto... y te ESCONDE que el ajuste es malo")

---
## 2. La matriz de covarianza

`curve_fit` devuelve `popt` (los parámetros) y `pcov` (su matriz de covarianza). Para $p$ parámetros
es una matriz $p\\times p$:

- la **diagonal** contiene las varianzas: $\\sigma_{a_i} = \\sqrt{\\mathrm{pcov}[i,i]}$;
- los **elementos fuera de la diagonal** contienen las covarianzas, que dicen cuán correlacionados
  están los parámetros entre sí.

La correlación normalizada es
$\\rho_{ij} = \\mathrm{pcov}[i,j] / \\sqrt{\\mathrm{pcov}[i,i]\\,\\mathrm{pcov}[j,j]}$, y va de $-1$ a $+1$.

Esto importa por una razón muy concreta: si vas a **combinar** los parámetros en una cuenta
posterior (por ejemplo $g = 4\\pi^2/a$, o $\\omega_0^2 = \\omega^2 + 1/\\tau^2$), propagar tratándolos
como independientes es incorrecto cuando $|\\rho|$ es grande.

In [ ]:
def matriz_correlacion(pcov):
    d = np.sqrt(np.diag(pcov))
    return pcov / np.outer(d, d)

print("pcov =\n", c_con)
print("\nmatriz de correlación =\n", np.round(matriz_correlacion(c_con), 3))
print(f"\nρ(a, b) = {matriz_correlacion(c_con)[0,1]:.3f}")

En un ajuste lineal la pendiente y la ordenada están casi siempre fuertemente anticorreladas: si
subís una, tenés que bajar la otra para seguir pasando por la nube de puntos. Un truco clásico para
descorrelacionarlas es ajustar $y = a(x - \\bar{x}) + b'$, centrando la variable independiente.

> **Ejercicio 6.1.** Ajustá con `recta(x - masa.mean(), a, b)` y volvé a calcular $\\rho$. ¿Bajó?
> ¿Cambió la pendiente?

---
## 3. El χ² reducido

Definimos

$$ \\chi^2 = \\sum_i \\frac{[y_i - f(x_i)]^2}{\\sigma_i^2}, \\qquad
   \\chi^2_\\nu = \\frac{\\chi^2}{\\nu}, \\qquad \\nu = N - p $$

donde $\\nu$ son los **grados de libertad**: cantidad de datos menos cantidad de parámetros
ajustados.

La idea es simple y potente: si el modelo es correcto y las barras de error están bien estimadas,
cada punto debería estar típicamente a una barra de error de la curva. Entonces cada término de la
suma vale aproximadamente 1, y $\\chi^2_\\nu \\approx 1$.

| valor | interpretación |
|---|---|
| $\\chi^2_\\nu \\approx 1$ | consistente con buen modelo **y** incertezas bien estimadas |
| $\\chi^2_\\nu \\gg 1$ | modelo inadecuado **o** incertezas subestimadas |
| $\\chi^2_\\nu \\ll 1$ | incertezas **sobre**estimadas (o parámetros de más) |

El tercer caso casi nunca se enseña y es muy formativo: un ajuste "demasiado bueno" no es un
triunfo, es una señal de que le pusiste barras de error más grandes de lo que corresponde. Y como
las barras infladas se propagan a los parámetros, estás reportando una incerteza final falsamente
grande.

**Atención al alcance:** $\\chi^2_\\nu$ solo tiene sentido si tenés incertezas $\\sigma_i$ que
estimaste independientemente. No sirve inventarlas a partir del propio ajuste — eso es circular.

In [ ]:
def chi2_reducido(y, y_modelo, sigma, n_parametros, verbose=True):
    """χ² reducido y su p-valor. sigma: incertezas experimentales reales."""
    y, y_modelo, sigma = map(lambda v: np.asarray(v, float), (y, y_modelo, sigma))
    chi2 = np.sum(((y - y_modelo) / sigma)**2)
    nu = len(y) - n_parametros
    chi2r = chi2 / nu
    p = stats.chi2.sf(chi2, nu)      # probabilidad de obtener un χ² igual o peor
    if verbose:
        if chi2r > 2:      lect = "modelo inadecuado o incertezas subestimadas"
        elif chi2r < 0.5:  lect = "incertezas probablemente sobreestimadas"
        else:              lect = "consistente con un buen ajuste"
        print(f"χ²  = {chi2:.2f}   ν = {nu}   χ²_ν = {chi2r:.2f}   p = {p:.3f}")
        print(f"  -> {lect}")
    return chi2r, p


chi2_reducido(elong, recta(masa, *p_con), err, n_parametros=2)

El **p-valor** es la probabilidad de obtener, por puro azar, un $\\chi^2$ igual o peor que el
observado, si el modelo fuese correcto. Un $p$ muy chico (digamos $< 0{,}01$) dice que el desacuerdo
difícilmente sea casualidad. Un $p$ muy cercano a 1 es la contraparte del $\\chi^2_\\nu \\ll 1$:
sospechosamente bueno.

### Los tres escenarios, lado a lado

Generamos datos de una recta con incerteza verdadera $\\sigma = 0{,}5$ y los ajustamos declarando
tres incertezas distintas.

In [ ]:
xg = np.linspace(0, 10, 15)
sigma_verdadero = 0.5
yg = 2.0*xg + 1.0 + np.random.normal(0, sigma_verdadero, len(xg))

casos = [("σ declarado correcto (0,50)", np.full_like(xg, 0.50)),
         ("σ subestimado (0,15)",        np.full_like(xg, 0.15)),
         ("σ sobreestimado (2,00)",      np.full_like(xg, 2.00))]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, (nombre, sg) in zip(axes, casos):
    pp, cc = curve_fit(recta, xg, yg, sigma=sg, absolute_sigma=True)
    c2r, pv = chi2_reducido(yg, recta(xg, *pp), sg, 2, verbose=False)
    ax.errorbar(xg, yg, yerr=sg, fmt='o', ms=4, capsize=3)
    ax.plot(xg, recta(xg, *pp), 'crimson', lw=1.6)
    ax.set_title(f'{nombre}\n$\\chi^2_\\nu$ = {c2r:.2f}   (p = {pv:.3f})', fontsize=10)
    ax.set_xlabel('x'); ax.grid(alpha=0.3)
    print(f"{nombre:<32} a = {pp[0]:.3f} ± {np.sqrt(cc[0,0]):.3f}")
axes[0].set_ylabel('y')
fig.tight_layout(); plt.show()

Los tres ajustes dan **la misma pendiente** —los pesos son uniformes, así que el mínimo es el mismo—
pero **incertezas del parámetro completamente distintas** y $\\chi^2_\\nu$ que delatan el problema.

Es un buen recordatorio de que la incerteza que reportás no sale del ajuste: sale de tu honestidad
al declarar las barras de error.

---
## 4. El promedio ponderado **es** un ajuste

Caso que aparece cada vez que medís lo mismo por dos métodos: tenés $N$ mediciones independientes de
una misma magnitud, cada una con su incerteza, y son compatibles entre sí. Tirar una es desperdiciar
información. ¿Cómo se combinan?

La respuesta habitual es una fórmula que se aprende de memoria. Pero no hace falta: **es el ajuste
por cuadrados mínimos ponderados que acabás de hacer, con el modelo más simple posible**, una
constante.

Ajustar $f(x) = c$ significa minimizar

$$ \chi^2(c) = \sum_i \frac{(x_i - c)^2}{\sigma_i^2} $$

Derivando respecto de $c$ e igualando a cero:

$$ \sum_i \frac{x_i - c}{\sigma_i^2} = 0 \quad\Longrightarrow\quad
   c = \frac{\sum x_i/\sigma_i^2}{\sum 1/\sigma_i^2}, \qquad
   \sigma_c = \left(\sum \frac{1}{\sigma_i^2}\right)^{-1/2} $$

Los pesos son los mismos $1/\sigma_i^2$ del ajuste ponderado. No hay nada nuevo que memorizar.

In [ ]:
def promedio_ponderado(x, sigma):
    '''Combinación de mediciones independientes. Devuelve (valor, incerteza, chi2_reducido).'''
    x, sigma = np.asarray(x, float), np.asarray(sigma, float)
    w = 1/sigma**2
    xp = np.sum(w*x)/np.sum(w)
    sp = 1/np.sqrt(np.sum(w))
    chi2 = np.sum(w*(x - xp)**2)          # consistencia interna de las mediciones combinadas
    return xp, sp, chi2/(len(x)-1)

# Verificación de que es literalmente el mismo ajuste
def constante(x, c):
    return np.full_like(np.asarray(x, float), c)

# (a) Dos mediciones de muy distinta calidad: el punto sobre los pesos
xs = np.array([10.00, 10.03])
ss = np.array([0.50, 0.02])
xp, sp, c2r = promedio_ponderado(xs, ss)
pc, cc = curve_fit(constante, np.arange(len(xs)), xs, sigma=ss, absolute_sigma=True, p0=[10.0])

print("(a) dos mediciones de muy distinta calidad")
print(f"    promedio simple                : {xs.mean():.5f}")
print(f"    promedio ponderado (fórmula)   : {xp:.5f} ± {sp:.5f}")
print(f"    curve_fit, modelo constante    : {pc[0]:.5f} ± {np.sqrt(cc[0,0]):.5f}   <- idéntico")
print(f"    peso relativo de la más precisa: {(ss[0]/ss[1])**2:.0f} veces")
print(f"    ¿la incerteza combinada es menor que la menor individual? "
      f"{sp:.5f} < {ss.min():.5f} -> {sp < ss.min()}")

# (b) Tres determinaciones de g, consistentes entre sí
print("\n(b) tres determinaciones independientes de g")
gs = np.array([9.79, 9.83, 9.80]); sg = np.array([0.05, 0.03, 0.02])
gp, sgp, c2g = promedio_ponderado(gs, sg)
print(f"    g combinado = {gp:.4f} ± {sgp:.4f} m/s²   χ²_ν = {c2g:.2f}  (ν = {len(gs)-1})")

# (c) Todas las sigma iguales: reaparece el error de la media
print("\n(c) todas las σ iguales -> σ/√N")
xi = np.array([2.10, 2.14, 2.09, 2.13, 2.12]); si = np.full(5, 0.04)
xip, sip, _ = promedio_ponderado(xi, si)
print(f"    ponderado : {xip:.4f} ± {sip:.5f}")
print(f"    σ/√N      : {0.04/np.sqrt(5):.5f}   <- el mismo número de la Clase 3")

# (d) Dos mediciones INCOMPATIBLES: el χ²_ν avisa
print("\n(d) dos mediciones incompatibles")
xb = np.array([9.62, 9.81]); sb = np.array([0.02, 0.03])
xbp, sbp, c2b = promedio_ponderado(xb, sb)
print(f"    'combinado' = {xbp:.4f} ± {sbp:.4f}   χ²_ν = {c2b:.1f}  <- NO reportar este número")

El promedio simple da 10,015, a mitad de camino entre las dos. El ponderado da prácticamente 10,03:
la medición de $\pm 0{,}02$ pesa **625 veces** más que la de $\pm 0{,}5$.

Tres consecuencias que conviene ver con números y no de palabra:

1. **La medición más precisa domina cuadráticamente.** Bajar la incerteza a la mitad multiplica el
   peso por cuatro. Por eso una sola medición buena puede volver irrelevantes a diez mediocres.
2. **La incerteza combinada es menor que la menor individual.** Combinar nunca empeora — siempre que
   las mediciones sean compatibles, que es la condición del punto 3.
3. **$\sigma/\sqrt{N}$ es el caso particular** en que todas las $\sigma_i$ son iguales. El error de la
   media que usás desde la Clase 3 no es una fórmula aparte: es esto, con pesos uniformes.

### Y de yapa, el test de consistencia

Como es un ajuste, tiene $\chi^2_\nu$, y ese número tiene una lectura precisa: con $\nu = N-1$ grados
de libertad, mide **si las mediciones que estás combinando son compatibles entre sí**.

Si $\chi^2_\nu \gg 1$, las mediciones no son consistentes y combinarlas no significa nada: el número
que sale no estima ninguna magnitud física, sino un promedio de cosas distintas. Ésa es la
diferencia entre combinar y licuar.

> **Ejercicio 6.6.** Combiná las dos determinaciones del período del faro del Colab 03 (las 50
> mediciones de un destello y las 8 tandas de 50). Mirá el $\chi^2_\nu$: ¿son consistentes? ¿Cuánto
> mejora la incerteza combinada respecto de la mejor de las dos? ¿Te sorprende cuán poco?

> **Ejercicio 6.7.** Ahora combiná dos mediciones deliberadamente incompatibles, por ejemplo
> $9{,}62 \pm 0{,}02$ y $9{,}81 \pm 0{,}03$. ¿Qué valor devuelve `promedio_ponderado`? ¿Qué dice el
> $\chi^2_\nu$? Escribí en una oración por qué el primer número no hay que reportarlo.

### Una advertencia metodológica que conviene tener escrita

En este notebook obtuviste dos valores de $k$: el del ajuste sin pesos y el del ponderado. Es
tentador aplicarles el test de compatibilidad de la Clase 3, o directamente combinarlos con
`promedio_ponderado`. **No se hace, y no es una cuestión de prolijidad.**

Los dos tests suponen mediciones **independientes**. Acá los dos números salen de los **mismos**
datos analizados de dos maneras distintas: están fuertemente correlacionados. En consecuencia
$\sqrt{\sigma_1^2+\sigma_2^2}$ sobreestima la incerteza de la diferencia, el test de compatibilidad
siempre daría "compatibles", y el promedio ponderado devolvería una incerteza absurdamente pequeña.

**Comparar dos análisis del mismo conjunto no es lo mismo que comparar dos mediciones.** Cuando
querés decidir cuál de dos análisis es el correcto, la herramienta no es la compatibilidad: es el
$\chi^2_\nu$ y los residuos de cada uno.

El caso legítimo aparece en la Clase 9, donde el $k$ estático (elongación bajo carga) y el $k$
dinámico (período de oscilación) salen de experimentos que no comparten ni un solo dato.

---
## 5. Ejercicios

**6.8.** Con tus datos del resorte y tus barras de error reales, hacé el ajuste ponderado. Reportá
$k$ con su incerteza y el $\chi^2_\nu$ **con su ν**. Si te da mayor que 3, no lo escondas:
preguntate si el modelo es incompleto o si subestimaste el error de lectura.

**6.9.** Multiplicá todas tus barras de error por 2 y volvé a ajustar. ¿Cambia $k$? ¿Cambia
$\sigma_k$? ¿Cambia $\chi^2_\nu$? Explicá cada respuesta por separado — las tres son distintas y las
tres importan.

**6.10.** Repetí el Ejercicio 6.9 pero con `absolute_sigma=False`. ¿Cuál de las tres respuestas
cambia? Ésa es exactamente la razón por la que el default de scipy destruye el diagnóstico.

**6.11.** Tomá el conjunto III del cuarteto de Anscombe (Colab 05), asignale barras de error de 0,5 a
todos los puntos y calculá $\chi^2_\nu$. ¿Detecta el problema que $R$ no detectaba?

**6.12.** *(criterio)* Un ajuste da $\chi^2_\nu = 0{,}08$ con $\nu = 12$. El estudiante concluye que
el modelo es excelente. ¿Qué le contestarías?

**6.13.** *(composición)* Medí dos resortes en **serie** y en **paralelo**. Predecí $k_{serie}$ y
$k_{paralelo}$ a partir de $k_1$ y $k_2$ **con su incerteza propagada**, y contrastá contra la
medición directa. Observación de diseño que sale gratis: la incerteza relativa del paralelo predicho
es menor que la de cada resorte por separado, y la del serie está dominada por el más blando.
¿Por qué?

**6.14.** *(optativo — σ en la variable independiente)* En el Colab 04 usaste el criterio
$\sigma_x|dy/dx|$ contra $\sigma_y$ para decidir si la incerteza en $x$ era despreciable. Aplicalo a
tus datos del resorte: ¿cuánto vale la incerteza de la masa colgada, y qué efecto tiene sobre la
elongación? Lo obligatorio es **verificar la hipótesis**, no dominar `scipy.odr`.